### Out-of-sample Model Building

In [1]:
import os
os.getcwd()
# os.chdir(path)    # or you can set your working dir.

'/Users/xingfuxu/PycharmProjects/EquityPremiumPrediction3'

In [2]:
# Your working dir should include "NN_models.py", Perform_CW_test.py" and "Perform_PT_test.py" files.
from Perform_CW_test import CW_test
from Perform_PT_test import PT_test
from NN_models import Net3

In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso, ElasticNet
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
import torch
from tqdm import tqdm
#
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import AdaBoostRegressor
from xgboost import XGBRegressor
#
from sklearn.metrics import mean_squared_error

In [4]:
# set seed
torch.manual_seed(1)
np.random.seed(1)

# read data
predictor_df = pd.read_csv('result_predictor.csv')
predictor_df.head()

,month,log_equity_premium,equity_premium,DP,DY,EP,SVAR,BM,NTIS,TBL,...,MA_2_9,MA_2_12,MA_3_9,MA_3_12,MOM_1,MOM_2,MOM_3,MOM_6,MOM_9,MOM_12
0,192701,-0.005710,-0.00571,-2.942374,-2.963349,-2.374773,0.00047,0.44371,0.05082,3.23,...,1,1,1,1,0,0,1,1,1,1
1,192702,0.042017,0.04302,-2.979535,-2.932946,-2.430353,0.00029,0.42850,0.05167,3.29,...,1,1,1,1,1,1,1,1,1,1
2,192703,0.004697,0.00472,-2.976535,-2.970053,-2.445079,0.00092,0.46977,0.04636,3.20,...,1,1,1,1,1,1,1,1,1,1
3,192704,0.009940,0.01002,-2.984225,-2.967143,-2.471309,0.00060,0.45675,0.05051,3.39,...,1,1,1,1,1,1,1,1,1,1
4,192705,0.057987,0.05985,-3.025963,-2.975058,-2.531446,0.00039,0.43478,0.05528,3.33,...,1,1,1,1,1,1,1,1,1,1


In [5]:
# remove irrelavent columns
predictor0 = predictor_df.drop(['month', 'equity_premium'], axis=1)
# get all the predictors and set the log equity premium 1-month ahead
predictor = np.concatenate([predictor0['log_equity_premium'][1:].values.reshape(-1, 1),
                            predictor0.iloc[0:(predictor0.shape[0] - 1), 1:]], axis=1)

# number of rows
N = predictor.shape[0]

# number of all columns, including the log equity premium
n_cols = predictor.shape[1]


# Actual one-month ahead log equity premium
actual = predictor[:, [0]]

# Historical average forecasting as benchmark
y_pred_HA = predictor0['log_equity_premium'].values[0:(predictor0.shape[0] - 1), ].cumsum() / np.arange(1, N + 1)
y_pred_HA = y_pred_HA.reshape(-1, 1)

In [6]:
## Out-of-sample: 1957:01-2020:12
in_out_1957 = predictor_df.index[predictor_df['month'] == 195701][0]
actual_1957 = actual[in_out_1957:, ]
y_pred_HA_1957 = y_pred_HA[in_out_1957:, ]
MSFE_HA_1957 = mean_squared_error(y_pred_HA_1957, actual_1957)

# Machine Learning methods used in GKX (2020)
y_pred_OLS_1957, y_pred_PLS_1957, y_pred_PCR_1957,  y_pred_LASSO_1957 = [], [], [], []
y_pred_ENet_1957, y_pred_GBRT_1957, y_pred_RF_1957, y_pred_NN3_1957 = [], [], [], []

## Other commonly used machine learning method
y_pred_SVR_1957, y_pred_KNR_1957, y_pred_AdaBoost_1957, y_pred_XGBoost_1957 = [], [], [], []
y_pred_combination_1957 = []

In [7]:
# control the update month of models during out-of-sample period (GKX, 2020). 
year_index = 1  # Our models will be updated at month 1, 13, 25, ...

In [8]:
for t in tqdm(range(in_out_1957, N)):
    #
    X_train_all = predictor[:t, 1:n_cols]
    y_train_all = predictor[:t, 0]
    #
    X_train = X_train_all[0:int(len(X_train_all) * 0.85), :]
    X_validation = X_train_all[int(len(X_train_all) * 0.85):t, :]
    y_train = y_train_all[0:int(len(X_train_all) * 0.85)]
    y_validation = y_train_all[int(len(X_train_all) * 0.85):t]
    #
    if year_index % 12 == 1:
        year_index += 1
        # OLS
        OLS = LinearRegression()
        OLS.fit(X_train_all, y_train_all)
        y_pred_OLS_1957.append(OLS.predict(predictor[[t], 1:n_cols]))

        # PLS
        PLS_param = [1, 2, 3, 4, 5, 6, 7, 8]
        PLS_result = {}
        for k in PLS_param:
            PLS = PLSRegression(n_components=k)
            PLS.fit(X_train, y_train)
            mse = mean_squared_error(PLS.predict(X_validation), y_validation)
            PLS_result[mse] = k
        PLS_best_param = PLS_result[min(PLS_result.keys())]
        PLS_model = PLSRegression(n_components=PLS_best_param)
        PLS_model.fit(X_train_all, y_train_all)
        y_pred_PLS_1957.append(PLS_model.predict(predictor[[t], 1:n_cols]))

        # PCR
        PCR_param = [1, 2, 3, 4, 5, 6, 7, 8]
        PCR_result = {}
        for k in PCR_param:
            pca = PCA(n_components=k)
            pca.fit(X_train)
            comps = pca.transform(X_train)
            forecast = LinearRegression()
            forecast.fit(comps, y_train)
            mse = mean_squared_error(forecast.predict(pca.transform(X_validation)), y_validation)
            PCR_result[mse] = k
        #
        PCR_best_param = PCR_result[min(PCR_result.keys())]
        PCR_model = PCA(n_components=PCR_best_param)
        PCR_model.fit(X_train_all)
        PCR_comps = PCR_model.transform(X_train_all)
        PCR_forecast = LinearRegression()
        PCR_forecast.fit(PCR_comps, y_train_all)
        y_pred_PCR_1957.append(PCR_forecast.predict(PCR_model.transform(predictor[[t], 1:n_cols])))

        # LASSO
        LASSO_param = 10 ** np.arange(-4, -1 + 0.001, 0.1)
        LASSO_result = {}
        for alpha in LASSO_param:
            LASSO = Lasso(alpha=alpha)
            LASSO.fit(X_train, y_train)
            mse = mean_squared_error(LASSO.predict(X_validation), y_validation)
            LASSO_result[mse] = alpha
        LASSO_best_param = LASSO_result[min(LASSO_result.keys())]
        LASSO_model = Lasso(alpha=LASSO_best_param)
        LASSO_model.fit(X_train_all, y_train_all)
        y_pred_LASSO_1957.append(LASSO_model.predict(predictor[[t], 1:n_cols]))

        # ENet
        ENet_param = 10 ** np.arange(-4, -1 + 0.001, 0.1)
        ENet_result = {}
        for alpha in ENet_param:
            ENet = ElasticNet(alpha=alpha, l1_ratio=0.5)
            ENet.fit(X_train, y_train)
            mse = mean_squared_error(ENet.predict(X_validation), y_validation)
            ENet_result[mse] = alpha
        ENet_best_param = ENet_result[min(ENet_result.keys())]
        ENet_model = ElasticNet(alpha=ENet_best_param, l1_ratio=0.5)
        ENet_model.fit(X_train_all, y_train_all)
        y_pred_ENet_1957.append(ENet_model.predict(predictor[[t], 1:n_cols]))

        # GBRT
        GBRT_param = [1, 2, 3, 4, 5, 6, 7, 8]
        GBRT_result = {}
        for depth in GBRT_param:
            GBRT = GradientBoostingRegressor(max_depth=depth)
            GBRT.fit(X_train, y_train)
            mse = mean_squared_error(GBRT.predict(X_validation), y_validation)
            GBRT_result[mse] = depth
        GBRT_best_param = GBRT_result[min(GBRT_result.keys())]
        GBRT_model = GradientBoostingRegressor(max_depth=GBRT_best_param)
        GBRT_model.fit(X_train_all, y_train_all)
        y_pred_GBRT_1957.append(GBRT_model.predict(predictor[[t], 1:n_cols]))

        # RF
        RF_param =[1, 2, 3, 4, 5, 6, 7, 8]
        RF_result = {}
        for depth in RF_param:
            RF = RandomForestRegressor(max_depth=depth)
            RF.fit(X_train, y_train)
            mse = mean_squared_error(RF.predict(X_validation), y_validation)
            RF_result[mse] = depth
        RF_best_param = RF_result[min(RF_result.keys())]
        RF_model = RandomForestRegressor(max_depth=RF_best_param)
        RF_model.fit(X_train_all, y_train_all)
        y_pred_RF_1957.append(RF_model.predict(predictor[[t], 1:n_cols]))


        # NN3
        X_train_tensor = torch.tensor(X_train, dtype=torch.float)
        y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float)
        X_validation_tensor = torch.tensor(X_validation, dtype=torch.float)
        y_validation_tensor = torch.tensor(y_validation, dtype=torch.float)
        NN3_l2_param = 10 ** np.arange(-5, -3 + 0.0001, 0.1)
        NN3_result = {}
        NN3 = Net3(n_cols - 1, 32, 16, 8, 1)

        for l2 in NN3_l2_param:
            optimizer = torch.optim.SGD(NN3.parameters(), lr=0.01, weight_decay=l2)
            loss_func = torch.nn.MSELoss()
            for i in range(100):
                out = NN3(X_train_tensor)
                loss = loss_func(out, y_train_tensor)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            mse = mean_squared_error(NN3(X_validation_tensor).detach().numpy(), y_validation)
            NN3_result[mse] = l2
        NN3_best_param = NN3_result[min(NN3_result.keys())]
        NN3_optimizer = torch.optim.SGD(NN3.parameters(), lr=0.01, weight_decay=NN3_best_param)
        NN3_loss_func = torch.nn.MSELoss()
        X_train_all_tensor = torch.tensor(X_train_all, dtype=torch.float)
        y_train_all_tensor = torch.tensor(y_train_all.reshape(-1, 1), dtype=torch.float)
        for i in range(100):
            NN3_out = NN3(X_train_all_tensor)
            NN3_loss = NN3_loss_func(NN3_out, y_train_all_tensor)
            NN3_optimizer.zero_grad()
            NN3_loss.backward()
            NN3_optimizer.step()
        y_pred_NN3_1957.append(NN3(torch.tensor(predictor[[t], 1:n_cols],
                                                dtype=torch.float)).detach().numpy()[0])
        ## Other commmonly used ML methods
        # SVR
        SVR_param = ['linear', 'poly', 'rbf', 'sigmoid']
        SVR_result = {}
        for kernel in SVR_param:
            SVR_tmp = SVR(kernel=kernel)
            SVR_tmp.fit(X_train, y_train)
            mse = mean_squared_error(SVR_tmp.predict(X_validation), y_validation)
            SVR_result[mse] = kernel
        SVR_best_param = SVR_result[min(SVR_result.keys())]
        SVR_model = SVR(kernel=SVR_best_param)
        SVR_model.fit(X_train_all, y_train_all)
        y_pred_SVR_1957.append(SVR_model.predict(predictor[[t], 1:n_cols]))

        # KNR
        KNR = KNeighborsRegressor()
        KNR_param = [5, 10, 20, 25, 30, 40, 50, 60, 70]
        KNR_result = {}
        for n_neighbors in KNR_param:
            KNR = KNeighborsRegressor(n_neighbors=n_neighbors)
            KNR.fit(X_train, y_train)
            mse = mean_squared_error(KNR.predict(X_validation), y_validation)
            KNR_result[mse] = n_neighbors
        KNR_best_param = KNR_result[min(KNR_result.keys())]
        KNR_model = KNeighborsRegressor(n_neighbors=KNR_best_param)
        KNR_model.fit(X_train_all, y_train_all)
        y_pred_KNR_1957.append(KNR_model.predict(predictor[[t], 1:n_cols]))

        # AdaBoost
        AdaBoost_param = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
        AdaBoost_result = {}
        for n_estimators in AdaBoost_param:
            AdaBoost = AdaBoostRegressor(n_estimators=n_estimators)
            AdaBoost.fit(X_train, y_train)
            mse = mean_squared_error(AdaBoost.predict(X_validation), y_validation)
            AdaBoost_result[mse] = n_estimators
        AdaBoost_best_param = AdaBoost_result[min(AdaBoost_result.keys())]
        AdaBoost_model = AdaBoostRegressor(n_estimators=AdaBoost_best_param)
        AdaBoost_model.fit(X_train_all, y_train_all)
        y_pred_AdaBoost_1957.append(AdaBoost_model.predict(predictor[[t], 1:n_cols]))

        # XGBoost
        XGBoost_param = [1, 2, 3, 4, 5, 6, 7, 8]
        XGBoost_result = {}
        for max_depth in XGBoost_param:
            XGBoost = XGBRegressor(max_depth=max_depth)
            XGBoost.fit(X_train, y_train)
            mse = mean_squared_error(XGBoost.predict(X_validation), y_validation)
            XGBoost_result[mse] = max_depth
        XGB_best_param = XGBoost_result[min(XGBoost_result.keys())]
        XGB_model = XGBRegressor(max_depth=XGB_best_param)
        XGB_model.fit(X_train_all, y_train_all)
        y_pred_XGBoost_1957.append(XGB_model.predict(predictor[[t], 1:n_cols]))
    else:
        year_index += 1
        y_pred_OLS_1957.append(OLS.predict(predictor[[t], 1:n_cols]))
        y_pred_PLS_1957.append(PLS_model.predict(predictor[[t], 1:n_cols]))
        y_pred_PCR_1957.append(PCR_forecast.predict(PCR_model.transform(predictor[[t], 1:n_cols])))
        y_pred_LASSO_1957.append(LASSO_model.predict(predictor[[t], 1:n_cols]))
        y_pred_ENet_1957.append(ENet_model.predict(predictor[[t], 1:n_cols]))
        y_pred_GBRT_1957.append(GBRT_model.predict(predictor[[t], 1:n_cols]))
        y_pred_RF_1957.append(RF_model.predict(predictor[[t], 1:n_cols]))
        y_pred_NN3_1957.append(NN3(torch.tensor(predictor[[t], 1:n_cols], dtype=torch.float)).detach().numpy()[0])
        # Other commmonly used ML methods
        y_pred_SVR_1957.append(SVR_model.predict(predictor[[t], 1:n_cols]))
        y_pred_KNR_1957.append(KNR_model.predict(predictor[[t], 1:n_cols]))
        y_pred_AdaBoost_1957.append(AdaBoost_model.predict(predictor[[t], 1:n_cols]))
        y_pred_XGBoost_1957.append(XGB_model.predict(predictor[[t], 1:n_cols]))




100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 767/767 [08:10<00:00,  1.56it/s]


In [9]:
# Performance compared with HA
# OLS
y_pred_OLS_1957 = np.array(y_pred_OLS_1957).reshape(-1, 1)
MSFE_OLS_1957 = mean_squared_error(y_pred_OLS_1957, actual_1957)
OOS_R_OLS_1957 = 1 - MSFE_OLS_1957 / MSFE_HA_1957
MSFE_adjusted_OLS_1957, p_OLS_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_OLS_1957)
success_ratio_OLS_1957, PT_OLS_1957, p2_OLS_1957 = PT_test(actual_1957, y_pred_OLS_1957)
# PLS
y_pred_PLS_1957 = np.array(y_pred_PLS_1957).reshape(-1, 1)
MSFE_PLS_1957 = mean_squared_error(y_pred_PLS_1957, actual_1957)
OOS_R_PLS_1957 = 1 - MSFE_PLS_1957 / MSFE_HA_1957
MSFE_adjusted_PLS_1957, p_PLS_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_PLS_1957)
success_ratio_PLS_1957, PT_PLS_1957, p2_PLS_1957 = PT_test(actual_1957, y_pred_PLS_1957)
# PCR
y_pred_PCR_1957 = np.array(y_pred_PCR_1957).reshape(-1, 1)
MSFE_PCR_1957 = mean_squared_error(y_pred_PCR_1957, actual_1957)
OOS_R_PCR_1957 = 1 - MSFE_PCR_1957 / MSFE_HA_1957
MSFE_adjusted_PCR_1957, p_PCR_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_PCR_1957)
success_ratio_PCR_1957, PT_PCR_1957, p2_PCR_1957 = PT_test(actual_1957, y_pred_PCR_1957)
# LASSO
y_pred_LASSO_1957 = np.array(y_pred_LASSO_1957).reshape(-1, 1)
MSFE_LASSO_1957 = mean_squared_error(y_pred_LASSO_1957, actual_1957)
OOS_R_LASSO_1957 = 1 - MSFE_LASSO_1957 / MSFE_HA_1957
MSFE_adjusted_LASSO_1957, p_LASSO_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_LASSO_1957)
success_ratio_LASSO_1957, PT_LASSO_1957, p2_LASSO_1957 = PT_test(actual_1957, y_pred_LASSO_1957)
# ENet
y_pred_ENet_1957 = np.array(y_pred_ENet_1957).reshape(-1, 1)
MSFE_ENet_1957 = mean_squared_error(y_pred_ENet_1957, actual_1957)
OOS_R_ENet_1957 = 1 - MSFE_ENet_1957 / MSFE_HA_1957
MSFE_adjusted_ENet_1957, p_ENet_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_ENet_1957)
success_ratio_ENet_1957, PT_ENet_1957, p2_ENet_1957 = PT_test(actual_1957, y_pred_ENet_1957)
# GBRT
y_pred_GBRT_1957 = np.array(y_pred_GBRT_1957).reshape(-1, 1)
MSFE_GBRT_1957 = mean_squared_error(y_pred_GBRT_1957, actual_1957)
OOS_R_GBRT_1957 = 1 - MSFE_GBRT_1957 / MSFE_HA_1957
MSFE_adjusted_GBRT_1957, p_GBRT_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_GBRT_1957)
success_ratio_GBRT_1957, PT_GBRT_1957, p2_GBRT_1957 = PT_test(actual_1957, y_pred_GBRT_1957)
# RF
y_pred_RF_1957 = np.array(y_pred_RF_1957).reshape(-1, 1)
MSFE_RF_1957 = mean_squared_error(y_pred_RF_1957, actual_1957)
OOS_R_RF_1957 = 1 - MSFE_RF_1957 / MSFE_HA_1957
MSFE_adjusted_RF_1957, p_RF_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_RF_1957)
success_ratio_RF_1957, PT_RF_1957, p2_RF_1957 = PT_test(actual_1957, y_pred_RF_1957)
# NN3
y_pred_NN3_1957 = np.array(y_pred_NN3_1957).reshape(-1, 1)
MSFE_NN3_1957 = mean_squared_error(y_pred_NN3_1957, actual_1957)
OOS_R_NN3_1957 = 1 - MSFE_NN3_1957 / MSFE_HA_1957
MSFE_adjusted_NN3_1957, p_NN3_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_NN3_1957)
success_ratio_NN3_1957, PT_NN3_1957, p2_NN3_1957 = PT_test(actual_1957, y_pred_NN3_1957)
# SVR
y_pred_SVR_1957 = np.array(y_pred_SVR_1957).reshape(-1, 1)
MSFE_SVR_1957 = mean_squared_error(y_pred_SVR_1957, actual_1957)
OOS_R_SVR_1957 = 1 - MSFE_SVR_1957 / MSFE_HA_1957
MSFE_adjusted_SVR_1957, p_SVR_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_SVR_1957)
success_ratio_SVR_1957, PT_SVR_1957, p2_SVR_1957 = PT_test(actual_1957, y_pred_SVR_1957)
# KNR
y_pred_KNR_1957 = np.array(y_pred_KNR_1957).reshape(-1, 1)
MSFE_KNR_1957 = mean_squared_error(y_pred_KNR_1957, actual_1957)
OOS_R_KNR_1957 = 1 - MSFE_KNR_1957 / MSFE_HA_1957
MSFE_adjusted_KNR_1957, p_KNR_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_KNR_1957)
success_ratio_KNR_1957, PT_KNR_1957, p2_KNR_1957 = PT_test(actual_1957, y_pred_KNR_1957)
# AdaBoost
y_pred_AdaBoost_1957 = np.array(y_pred_AdaBoost_1957).reshape(-1, 1)
MSFE_AdaBoost_1957 = mean_squared_error(y_pred_AdaBoost_1957, actual_1957)
OOS_R_AdaBoost_1957 = 1 - MSFE_AdaBoost_1957 / MSFE_HA_1957
MSFE_adjusted_AdaBoost_1957, p_AdaBoost_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_AdaBoost_1957)
success_ratio_AdaBoost_1957, PT_AdaBoost_1957, p2_AdaBoost_1957 = PT_test(actual_1957, y_pred_AdaBoost_1957)
# XGBoost
y_pred_XGBoost_1957 = np.array(y_pred_XGBoost_1957).reshape(-1, 1)
MSFE_XGBoost_1957 = mean_squared_error(y_pred_XGBoost_1957, actual_1957)
OOS_R_XGBoost_1957 = 1 - MSFE_XGBoost_1957 / MSFE_HA_1957
MSFE_adjusted_XGBoost_1957, p_XGBoost_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_XGBoost_1957)
success_ratio_XGBoost_1957, PT_XGBoost_1957, p2_XGBoost_1957 = PT_test(actual_1957, y_pred_XGBoost_1957)
# Combination
y_pred_combination_1957 = np.concatenate([y_pred_OLS_1957, y_pred_PLS_1957, y_pred_PCR_1957, y_pred_LASSO_1957,
                                          y_pred_ENet_1957, y_pred_GBRT_1957, y_pred_RF_1957, y_pred_NN3_1957,
                                          y_pred_SVR_1957, y_pred_KNR_1957, y_pred_AdaBoost_1957,
                                          y_pred_XGBoost_1957], axis=1).mean(axis=1).reshape(-1, 1)
MSFE_combination_1957 = mean_squared_error(y_pred_combination_1957, actual_1957)
OOS_R_combination_1957 = 1 - MSFE_combination_1957 / MSFE_HA_1957
MSFE_adjusted_combination_1957, p_combination_1957 = CW_test(actual_1957, y_pred_HA_1957, y_pred_combination_1957)
success_ratio_combination_1957, PT_combination_1957, p2_combination_1957 = PT_test(actual_1957, y_pred_combination_1957)
# success ratio of HA
success_ratio_HA_1957, PT_HA_1957, p2_HA_1957 = PT_test(actual_1957, y_pred_HA_1957)

/Users/xingfuxu/PycharmProjects/EquityPremiumPrediction3/Perform_PT_test.py:36: RuntimeWarning: invalid value encountered in double_scalars
  stat = (p_hat - p_star) / np.sqrt(p_hat_var - p_star_var)


In [10]:
# Similarly, the codes below display results for out-of-sample period begining at 1990:01
#####################################################################
##################### Out-of-sample: 1990:01-2020:12 ################
#####################################################################
in_out_1990 = predictor_df.index[predictor_df['month'] == 199001][0]
actual_1990 = actual[in_out_1990:, ]
y_pred_HA_1990 = y_pred_HA[in_out_1990:, ]
MSFE_HA_1990 = mean_squared_error(y_pred_HA_1990, actual_1990)

# Machine Learning methods used in GKX (2020)
y_pred_OLS_1990, y_pred_PLS_1990, y_pred_PCR_1990,  y_pred_LASSO_1990 = [], [], [], []
y_pred_ENet_1990, y_pred_GBRT_1990, y_pred_RF_1990, y_pred_NN3_1990 = [], [], [], []

## Other commonly used machine learning method
y_pred_SVR_1990, y_pred_KNR_1990, y_pred_AdaBoost_1990, y_pred_XGBoost_1990 = [], [], [], []
y_pred_combination_1990 = []


year_index = 1   # control the update of model each year
for t in tqdm(range(in_out_1990, N)):
    #
    X_train_all = predictor[:t, 1:n_cols]
    y_train_all = predictor[:t, 0]
    #
    X_train = X_train_all[0:int(len(X_train_all) * 0.85), :]
    X_validation = X_train_all[int(len(X_train_all) * 0.85):t, :]
    y_train = y_train_all[0:int(len(X_train_all) * 0.85)]
    y_validation = y_train_all[int(len(X_train_all) * 0.85):t]
    #
    if year_index % 12 == 1:
        year_index += 1
        # OLS
        OLS = LinearRegression()
        OLS.fit(X_train_all, y_train_all)
        y_pred_OLS_1990.append(OLS.predict(predictor[[t], 1:n_cols]))

        # PLS
        PLS_param = [1, 2, 3, 4, 5, 6, 7, 8]
        PLS_result = {}
        for k in PLS_param:
            PLS = PLSRegression(n_components=k)
            PLS.fit(X_train, y_train)
            mse = mean_squared_error(PLS.predict(X_validation), y_validation)
            PLS_result[mse] = k
        PLS_best_param = PLS_result[min(PLS_result.keys())]
        PLS_model = PLSRegression(n_components=PLS_best_param)
        PLS_model.fit(X_train_all, y_train_all)
        y_pred_PLS_1990.append(PLS_model.predict(predictor[[t], 1:n_cols]))

        # PCR
        PCR_param = [1, 2, 3, 4, 5, 6, 7, 8]
        PCR_result = {}
        for k in PCR_param:
            pca = PCA(n_components=k)
            pca.fit(X_train)
            comps = pca.transform(X_train)
            forecast = LinearRegression()
            forecast.fit(comps, y_train)
            mse = mean_squared_error(forecast.predict(pca.transform(X_validation)), y_validation)
            PCR_result[mse] = k
        #
        PCR_best_param = PCR_result[min(PCR_result.keys())]
        PCR_model = PCA(n_components=PCR_best_param)
        PCR_model.fit(X_train_all)
        PCR_comps = PCR_model.transform(X_train_all)
        PCR_forecast = LinearRegression()
        PCR_forecast.fit(PCR_comps, y_train_all)
        y_pred_PCR_1990.append(PCR_forecast.predict(PCR_model.transform(predictor[[t], 1:n_cols])))

        # LASSO
        LASSO_param = 10 ** np.arange(-4, -1 + 0.001, 0.1)
        LASSO_result = {}
        for alpha in LASSO_param:
            LASSO = Lasso(alpha=alpha)
            LASSO.fit(X_train, y_train)
            mse = mean_squared_error(LASSO.predict(X_validation), y_validation)
            LASSO_result[mse] = alpha
        LASSO_best_param = LASSO_result[min(LASSO_result.keys())]
        LASSO_model = Lasso(alpha=LASSO_best_param)
        LASSO_model.fit(X_train_all, y_train_all)
        y_pred_LASSO_1990.append(LASSO_model.predict(predictor[[t], 1:n_cols]))

        # ENet
        ENet_param = 10 ** np.arange(-4, -1 + 0.001, 0.1)
        ENet_result = {}
        for alpha in ENet_param:
            ENet = ElasticNet(alpha=alpha, l1_ratio=0.5)
            ENet.fit(X_train, y_train)
            mse = mean_squared_error(ENet.predict(X_validation), y_validation)
            ENet_result[mse] = alpha
        ENet_best_param = ENet_result[min(ENet_result.keys())]
        ENet_model = ElasticNet(alpha=ENet_best_param, l1_ratio=0.5)
        ENet_model.fit(X_train_all, y_train_all)
        y_pred_ENet_1990.append(ENet_model.predict(predictor[[t], 1:n_cols]))

        # GBRT
        GBRT_param = [1, 2, 3, 4, 5, 6, 7, 8]
        GBRT_result = {}
        for depth in GBRT_param:
            GBRT = GradientBoostingRegressor(max_depth=depth)
            GBRT.fit(X_train, y_train)
            mse = mean_squared_error(GBRT.predict(X_validation), y_validation)
            GBRT_result[mse] = depth
        GBRT_best_param = GBRT_result[min(GBRT_result.keys())]
        GBRT_model = GradientBoostingRegressor(max_depth=GBRT_best_param)
        GBRT_model.fit(X_train_all, y_train_all)
        y_pred_GBRT_1990.append(GBRT_model.predict(predictor[[t], 1:n_cols]))

        # RF
        RF_param =[1, 2, 3, 4, 5, 6, 7, 8]
        RF_result = {}
        for depth in RF_param:
            RF = RandomForestRegressor(max_depth=depth)
            RF.fit(X_train, y_train)
            mse = mean_squared_error(RF.predict(X_validation), y_validation)
            RF_result[mse] = depth
        RF_best_param = RF_result[min(RF_result.keys())]
        RF_model = RandomForestRegressor(max_depth=RF_best_param)
        RF_model.fit(X_train_all, y_train_all)
        y_pred_RF_1990.append(RF_model.predict(predictor[[t], 1:n_cols]))


        # NN3
        X_train_tensor = torch.tensor(X_train, dtype=torch.float)
        y_train_tensor = torch.tensor(y_train.reshape(-1, 1), dtype=torch.float)
        X_validation_tensor = torch.tensor(X_validation, dtype=torch.float)
        y_validation_tensor = torch.tensor(y_validation, dtype=torch.float)
        NN3_l2_param = 10 ** np.arange(-5, -3 + 0.0001, 0.1)
        NN3_result = {}
        NN3 = Net3(n_cols - 1, 32, 16, 8, 1)

        for l2 in NN3_l2_param:
            optimizer = torch.optim.SGD(NN3.parameters(), lr=0.01, weight_decay=l2)
            loss_func = torch.nn.MSELoss()
            for i in range(100):
                out = NN3(X_train_tensor)
                loss = loss_func(out, y_train_tensor)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            mse = mean_squared_error(NN3(X_validation_tensor).detach().numpy(), y_validation)
            NN3_result[mse] = l2
        NN3_best_param = NN3_result[min(NN3_result.keys())]
        NN3_optimizer = torch.optim.SGD(NN3.parameters(), lr=0.01, weight_decay=NN3_best_param)
        NN3_loss_func = torch.nn.MSELoss()
        X_train_all_tensor = torch.tensor(X_train_all, dtype=torch.float)
        y_train_all_tensor = torch.tensor(y_train_all.reshape(-1, 1), dtype=torch.float)
        for i in range(100):
            NN3_out = NN3(X_train_all_tensor)
            NN3_loss = NN3_loss_func(NN3_out, y_train_all_tensor)
            NN3_optimizer.zero_grad()
            NN3_loss.backward()
            NN3_optimizer.step()
        y_pred_NN3_1990.append(NN3(torch.tensor(predictor[[t], 1:n_cols],
                                                dtype=torch.float)).detach().numpy()[0])
        ## Other commmonly used ML methods
        # SVR
        SVR_param = ['linear', 'poly', 'rbf', 'sigmoid']
        SVR_result = {}
        for kernel in SVR_param:
            SVR_tmp = SVR(kernel=kernel)
            SVR_tmp.fit(X_train, y_train)
            mse = mean_squared_error(SVR_tmp.predict(X_validation), y_validation)
            SVR_result[mse] = kernel
        SVR_best_param = SVR_result[min(SVR_result.keys())]
        SVR_model = SVR(kernel=SVR_best_param)
        SVR_model.fit(X_train_all, y_train_all)
        y_pred_SVR_1990.append(SVR_model.predict(predictor[[t], 1:n_cols]))

        # KNR
        KNR = KNeighborsRegressor()
        KNR_param = [5, 10, 20, 25, 30, 40, 50, 60, 70]
        KNR_result = {}
        for n_neighbors in KNR_param:
            KNR = KNeighborsRegressor(n_neighbors=n_neighbors)
            KNR.fit(X_train, y_train)
            mse = mean_squared_error(KNR.predict(X_validation), y_validation)
            KNR_result[mse] = n_neighbors
        KNR_best_param = KNR_result[min(KNR_result.keys())]
        KNR_model = KNeighborsRegressor(n_neighbors=KNR_best_param)
        KNR_model.fit(X_train_all, y_train_all)
        y_pred_KNR_1990.append(KNR_model.predict(predictor[[t], 1:n_cols]))

        # AdaBoost
        AdaBoost_param = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
        AdaBoost_result = {}
        for n_estimators in AdaBoost_param:
            AdaBoost = AdaBoostRegressor(n_estimators=n_estimators)
            AdaBoost.fit(X_train, y_train)
            mse = mean_squared_error(AdaBoost.predict(X_validation), y_validation)
            AdaBoost_result[mse] = n_estimators
        AdaBoost_best_param = AdaBoost_result[min(AdaBoost_result.keys())]
        AdaBoost_model = AdaBoostRegressor(n_estimators=AdaBoost_best_param)
        AdaBoost_model.fit(X_train_all, y_train_all)
        y_pred_AdaBoost_1990.append(AdaBoost_model.predict(predictor[[t], 1:n_cols]))

        # XGBoost
        XGBoost_param = [1, 2, 3, 4, 5, 6, 7, 8]
        XGBoost_result = {}
        for max_depth in XGBoost_param:
            XGBoost = XGBRegressor(max_depth=max_depth)
            XGBoost.fit(X_train, y_train)
            mse = mean_squared_error(XGBoost.predict(X_validation), y_validation)
            XGBoost_result[mse] = max_depth
        XGB_best_param = XGBoost_result[min(XGBoost_result.keys())]
        XGB_model = XGBRegressor(max_depth=XGB_best_param)
        XGB_model.fit(X_train_all, y_train_all)
        y_pred_XGBoost_1990.append(XGB_model.predict(predictor[[t], 1:n_cols]))
    else:
        year_index += 1
        y_pred_OLS_1990.append(OLS.predict(predictor[[t], 1:n_cols]))
        y_pred_PLS_1990.append(PLS_model.predict(predictor[[t], 1:n_cols]))
        y_pred_PCR_1990.append(PCR_forecast.predict(PCR_model.transform(predictor[[t], 1:n_cols])))
        y_pred_LASSO_1990.append(LASSO_model.predict(predictor[[t], 1:n_cols]))
        y_pred_ENet_1990.append(ENet_model.predict(predictor[[t], 1:n_cols]))
        y_pred_GBRT_1990.append(GBRT_model.predict(predictor[[t], 1:n_cols]))
        y_pred_RF_1990.append(RF_model.predict(predictor[[t], 1:n_cols]))
        y_pred_NN3_1990.append(NN3(torch.tensor(predictor[[t], 1:n_cols], dtype=torch.float)).detach().numpy()[0])
        # Other commmonly used ML methods
        y_pred_SVR_1990.append(SVR_model.predict(predictor[[t], 1:n_cols]))
        y_pred_KNR_1990.append(KNR_model.predict(predictor[[t], 1:n_cols]))
        y_pred_AdaBoost_1990.append(AdaBoost_model.predict(predictor[[t], 1:n_cols]))
        y_pred_XGBoost_1990.append(XGB_model.predict(predictor[[t], 1:n_cols]))

100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 371/371 [04:40<00:00,  1.32it/s]


In [11]:
# Performance compared with HA
# OLS
y_pred_OLS_1990 = np.array(y_pred_OLS_1990).reshape(-1, 1)
MSFE_OLS_1990 = mean_squared_error(y_pred_OLS_1990, actual_1990)
OOS_R_OLS_1990 = 1 - MSFE_OLS_1990 / MSFE_HA_1990
MSFE_adjusted_OLS_1990, p_OLS_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_OLS_1990)
success_ratio_OLS_1990, PT_OLS_1990, p2_OLS_1990 = PT_test(actual_1990, y_pred_OLS_1990)
# PLS
y_pred_PLS_1990 = np.array(y_pred_PLS_1990).reshape(-1, 1)
MSFE_PLS_1990 = mean_squared_error(y_pred_PLS_1990, actual_1990)
OOS_R_PLS_1990 = 1 - MSFE_PLS_1990 / MSFE_HA_1990
MSFE_adjusted_PLS_1990, p_PLS_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_PLS_1990)
success_ratio_PLS_1990, PT_PLS_1990, p2_PLS_1990 = PT_test(actual_1990, y_pred_PLS_1990)
# PCR
y_pred_PCR_1990 = np.array(y_pred_PCR_1990).reshape(-1, 1)
MSFE_PCR_1990 = mean_squared_error(y_pred_PCR_1990, actual_1990)
OOS_R_PCR_1990 = 1 - MSFE_PCR_1990 / MSFE_HA_1990
MSFE_adjusted_PCR_1990, p_PCR_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_PCR_1990)
success_ratio_PCR_1990, PT_PCR_1990, p2_PCR_1990 = PT_test(actual_1990, y_pred_PCR_1990)
# LASSO
y_pred_LASSO_1990 = np.array(y_pred_LASSO_1990).reshape(-1, 1)
MSFE_LASSO_1990 = mean_squared_error(y_pred_LASSO_1990, actual_1990)
OOS_R_LASSO_1990 = 1 - MSFE_LASSO_1990 / MSFE_HA_1990
MSFE_adjusted_LASSO_1990, p_LASSO_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_LASSO_1990)
success_ratio_LASSO_1990, PT_LASSO_1990, p2_LASSO_1990 = PT_test(actual_1990, y_pred_LASSO_1990)
# ENet
y_pred_ENet_1990 = np.array(y_pred_ENet_1990).reshape(-1, 1)
MSFE_ENet_1990 = mean_squared_error(y_pred_ENet_1990, actual_1990)
OOS_R_ENet_1990 = 1 - MSFE_ENet_1990 / MSFE_HA_1990
MSFE_adjusted_ENet_1990, p_ENet_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_ENet_1990)
success_ratio_ENet_1990, PT_ENet_1990, p2_ENet_1990 = PT_test(actual_1990, y_pred_ENet_1990)
# GBRT
y_pred_GBRT_1990 = np.array(y_pred_GBRT_1990).reshape(-1, 1)
MSFE_GBRT_1990 = mean_squared_error(y_pred_GBRT_1990, actual_1990)
OOS_R_GBRT_1990 = 1 - MSFE_GBRT_1990 / MSFE_HA_1990
MSFE_adjusted_GBRT_1990, p_GBRT_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_GBRT_1990)
success_ratio_GBRT_1990, PT_GBRT_1990, p2_GBRT_1990 = PT_test(actual_1990, y_pred_GBRT_1990)
# RF
y_pred_RF_1990 = np.array(y_pred_RF_1990).reshape(-1, 1)
MSFE_RF_1990 = mean_squared_error(y_pred_RF_1990, actual_1990)
OOS_R_RF_1990 = 1 - MSFE_RF_1990 / MSFE_HA_1990
MSFE_adjusted_RF_1990, p_RF_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_RF_1990)
success_ratio_RF_1990, PT_RF_1990, p2_RF_1990 = PT_test(actual_1990, y_pred_RF_1990)
# NN3
y_pred_NN3_1990 = np.array(y_pred_NN3_1990).reshape(-1, 1)
MSFE_NN3_1990 = mean_squared_error(y_pred_NN3_1990, actual_1990)
OOS_R_NN3_1990 = 1 - MSFE_NN3_1990 / MSFE_HA_1990
MSFE_adjusted_NN3_1990, p_NN3_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_NN3_1990)
success_ratio_NN3_1990, PT_NN3_1990, p2_NN3_1990 = PT_test(actual_1990, y_pred_NN3_1990)
# SVR
y_pred_SVR_1990 = np.array(y_pred_SVR_1990).reshape(-1, 1)
MSFE_SVR_1990 = mean_squared_error(y_pred_SVR_1990, actual_1990)
OOS_R_SVR_1990 = 1 - MSFE_SVR_1990 / MSFE_HA_1990
MSFE_adjusted_SVR_1990, p_SVR_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_SVR_1990)
success_ratio_SVR_1990, PT_SVR_1990, p2_SVR_1990 = PT_test(actual_1990, y_pred_SVR_1990)
# KNR
y_pred_KNR_1990 = np.array(y_pred_KNR_1990).reshape(-1, 1)
MSFE_KNR_1990 = mean_squared_error(y_pred_KNR_1990, actual_1990)
OOS_R_KNR_1990 = 1 - MSFE_KNR_1990 / MSFE_HA_1990
MSFE_adjusted_KNR_1990, p_KNR_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_KNR_1990)
success_ratio_KNR_1990, PT_KNR_1990, p2_KNR_1990 = PT_test(actual_1990, y_pred_KNR_1990)
# AdaBoost
y_pred_AdaBoost_1990 = np.array(y_pred_AdaBoost_1990).reshape(-1, 1)
MSFE_AdaBoost_1990 = mean_squared_error(y_pred_AdaBoost_1990, actual_1990)
OOS_R_AdaBoost_1990 = 1 - MSFE_AdaBoost_1990 / MSFE_HA_1990
MSFE_adjusted_AdaBoost_1990, p_AdaBoost_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_AdaBoost_1990)
success_ratio_AdaBoost_1990, PT_AdaBoost_1990, p2_AdaBoost_1990 = PT_test(actual_1990, y_pred_AdaBoost_1990)
# XGBoost
y_pred_XGBoost_1990 = np.array(y_pred_XGBoost_1990).reshape(-1, 1)
MSFE_XGBoost_1990 = mean_squared_error(y_pred_XGBoost_1990, actual_1990)
OOS_R_XGBoost_1990 = 1 - MSFE_XGBoost_1990 / MSFE_HA_1990
MSFE_adjusted_XGBoost_1990, p_XGBoost_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_XGBoost_1990)
success_ratio_XGBoost_1990, PT_XGBoost_1990, p2_XGBoost_1990 = PT_test(actual_1990, y_pred_XGBoost_1990)
# Combination
y_pred_combination_1990 = np.concatenate([y_pred_OLS_1990, y_pred_PLS_1990, y_pred_PCR_1990, y_pred_LASSO_1990,
                                          y_pred_ENet_1990, y_pred_GBRT_1990, y_pred_RF_1990, y_pred_NN3_1990,
                                          y_pred_SVR_1990, y_pred_KNR_1990, y_pred_AdaBoost_1990,
                                          y_pred_XGBoost_1990], axis=1).mean(axis=1).reshape(-1, 1)
MSFE_combination_1990 = mean_squared_error(y_pred_combination_1990, actual_1990)
OOS_R_combination_1990 = 1 - MSFE_combination_1990 / MSFE_HA_1990
MSFE_adjusted_combination_1990, p_combination_1990 = CW_test(actual_1990, y_pred_HA_1990, y_pred_combination_1990)
success_ratio_combination_1990, PT_combination_1990, p2_combination_1990 = PT_test(actual_1990, y_pred_combination_1990)
# success ratio of HA
success_ratio_HA_1990, PT_HA_1990, p2_HA_1990 = PT_test(actual_1990, y_pred_HA_1990)

/Users/xingfuxu/PycharmProjects/EquityPremiumPrediction3/Perform_PT_test.py:36: RuntimeWarning: invalid value encountered in double_scalars
  stat = (p_hat - p_star) / np.sqrt(p_hat_var - p_star_var)


In [12]:
# output results
results_OOS_sample_forecast1 = np.array([
    [np.nan, np.nan, np.nan, success_ratio_HA_1957, PT_HA_1957, p2_HA_1957,
     np.nan, np.nan, np.nan, success_ratio_HA_1990, PT_HA_1990, p2_HA_1990],
    [OOS_R_OLS_1957, MSFE_adjusted_OLS_1957, p_OLS_1957, success_ratio_OLS_1957, PT_OLS_1957, p2_OLS_1957, 
     OOS_R_OLS_1990, MSFE_adjusted_OLS_1990, p_OLS_1990, success_ratio_OLS_1990, PT_OLS_1990, p2_OLS_1990],
    [OOS_R_PLS_1957, MSFE_adjusted_PLS_1957, p_PLS_1957, success_ratio_PLS_1957, PT_PLS_1957, p2_PLS_1957,
     OOS_R_PLS_1990, MSFE_adjusted_PLS_1990, p_PLS_1990, success_ratio_PLS_1990, PT_PLS_1990, p2_PLS_1990],
    [OOS_R_PCR_1957, MSFE_adjusted_PCR_1957, p_PCR_1957, success_ratio_PCR_1957, PT_PCR_1957, p2_PCR_1957,
     OOS_R_PCR_1990, MSFE_adjusted_PCR_1990, p_PCR_1990, success_ratio_PCR_1990, PT_PCR_1990, p2_PCR_1990],
    [OOS_R_LASSO_1957, MSFE_adjusted_LASSO_1957, p_LASSO_1957, success_ratio_LASSO_1957, PT_LASSO_1957, p2_LASSO_1957,
     OOS_R_LASSO_1990, MSFE_adjusted_LASSO_1990, p_LASSO_1990, success_ratio_LASSO_1990, PT_LASSO_1990, p2_LASSO_1990],
    [OOS_R_ENet_1957, MSFE_adjusted_ENet_1957, p_ENet_1957, success_ratio_ENet_1957, PT_ENet_1957, p2_ENet_1957, 
     OOS_R_ENet_1990, MSFE_adjusted_ENet_1990, p_ENet_1990, success_ratio_ENet_1990, PT_ENet_1990, p2_ENet_1990],
    [OOS_R_GBRT_1957, MSFE_adjusted_GBRT_1957, p_GBRT_1957, success_ratio_GBRT_1957, PT_GBRT_1957, p2_GBRT_1957,
     OOS_R_GBRT_1990, MSFE_adjusted_GBRT_1990, p_GBRT_1990, success_ratio_GBRT_1990, PT_GBRT_1990, p2_GBRT_1990],
    [OOS_R_RF_1957, MSFE_adjusted_RF_1957, p_RF_1957, success_ratio_RF_1957, PT_RF_1957, p2_RF_1957, 
     OOS_R_RF_1990, MSFE_adjusted_RF_1990, p_RF_1990, success_ratio_RF_1990, PT_RF_1990, p2_RF_1990],
    [OOS_R_NN3_1957, MSFE_adjusted_NN3_1957, p_NN3_1957, success_ratio_NN3_1957, PT_NN3_1957, p2_NN3_1957,
     OOS_R_NN3_1990, MSFE_adjusted_NN3_1990, p_NN3_1990, success_ratio_NN3_1990, PT_NN3_1990, p2_NN3_1990]
])
results_OOS_sample_forecast1 = pd.DataFrame(results_OOS_sample_forecast1)
results_OOS_sample_forecast1.columns = ['1957-' + s for s in ['R2', 'CW_stat', 'pValue', 'Success_ratio', 'PT-stat', 'p2Value']] + \
    ['1990-' + s for s in ['R2', 'CW_stat', 'pValue', 'Success_ratio', 'PT-stat', 'p2Value']]
results_OOS_sample_forecast1.insert(0, "Forecasting models",  ["HA", "OLS", "PLS", "PCR", "LASSO", 
                                                               "ENet", "GBRT", "RF", "NN3"])
results_OOS_sample_forecast1.to_csv("results_OOS_sample_forecast1.csv", index=False)
results_OOS_sample_forecast1

,Forecasting models,1957-R2,1957-CW_stat,1957-pValue,1957-Success_ratio,1957-PT-stat,1957-p2Value,1990-R2,1990-CW_stat,1990-pValue,1990-Success_ratio,1990-PT-stat,1990-p2Value
0,HA,NaN,NaN,NaN,0.599739,NaN,NaN,NaN,NaN,NaN,0.641509,NaN,NaN
1,OLS,-0.126798,0.613661,0.269720,0.560626,2.200000,0.013903,-0.115473,0.476050,0.317020,0.614555,2.399699,0.008204
2,PLS,-0.051884,0.597302,0.275153,0.555411,1.334480,0.091023,-0.084521,-0.151539,0.560225,0.544474,0.090032,0.464131
3,PCR,0.002305,1.840172,0.032872,0.591917,3.016182,0.001280,0.002854,0.990722,0.160911,0.609164,1.703533,0.044234
4,LASSO,-0.015953,0.725606,0.234040,0.589309,2.117765,0.017098,-0.047573,0.019927,0.492051,0.606469,0.049195,0.480382
5,ENet,-0.015759,0.729278,0.232916,0.588005,2.012708,0.022073,-0.047099,0.023914,0.490461,0.603774,-0.161365,0.564097
6,GBRT,-0.397314,0.622435,0.266828,0.550196,-0.042692,0.517027,-0.539149,0.317289,0.375512,0.598383,1.562194,0.059121
7,RF,-0.116530,-0.440730,0.670296,0.571056,-0.414004,0.660564,-0.368071,-2.038170,0.979234,0.579515,-0.656446,0.744232
8,NN3,-0.272533,0.391253,0.347805,0.546284,1.094319,0.136908,-0.234674,-0.605567,0.727599,0.536388,-0.089280,0.535570


In [13]:
#
results_OOS_sample_forecast2 = np.array([
    [np.nan, np.nan, np.nan, success_ratio_HA_1957, PT_HA_1957, p2_HA_1957,
     np.nan, np.nan, np.nan, success_ratio_HA_1990, PT_HA_1990, p2_HA_1990],
    [OOS_R_SVR_1957, MSFE_adjusted_SVR_1957, p_SVR_1957, success_ratio_SVR_1957, PT_SVR_1957, p2_SVR_1957,
     OOS_R_SVR_1990, MSFE_adjusted_SVR_1990, p_SVR_1990, success_ratio_SVR_1990, PT_SVR_1990, p2_SVR_1990],
    [OOS_R_KNR_1957, MSFE_adjusted_KNR_1957, p_KNR_1957, success_ratio_KNR_1957, PT_KNR_1957, p2_KNR_1957,
     OOS_R_KNR_1990, MSFE_adjusted_KNR_1990, p_KNR_1990, success_ratio_KNR_1990, PT_KNR_1990, p2_KNR_1990],
    [OOS_R_AdaBoost_1957, MSFE_adjusted_AdaBoost_1957, p_AdaBoost_1957, success_ratio_AdaBoost_1957, PT_AdaBoost_1957, p2_AdaBoost_1957,
     OOS_R_AdaBoost_1990, MSFE_adjusted_AdaBoost_1990, p_AdaBoost_1990, success_ratio_AdaBoost_1990, PT_AdaBoost_1990, p2_AdaBoost_1990],
    [OOS_R_XGBoost_1957, MSFE_adjusted_XGBoost_1957, p_XGBoost_1957, success_ratio_XGBoost_1957, PT_XGBoost_1957, p2_XGBoost_1957,
     OOS_R_XGBoost_1990, MSFE_adjusted_XGBoost_1990, p_XGBoost_1990, success_ratio_XGBoost_1990, PT_XGBoost_1990, p2_XGBoost_1990],
    [OOS_R_combination_1957, MSFE_adjusted_combination_1957, p_combination_1957, success_ratio_combination_1957, PT_combination_1957, p2_combination_1957,
     OOS_R_combination_1990, MSFE_adjusted_combination_1990, p_combination_1990, success_ratio_combination_1990, PT_combination_1990, p2_combination_1990]
])
#
results_OOS_sample_forecast2 = pd.DataFrame(results_OOS_sample_forecast2)
results_OOS_sample_forecast2.columns = ['1957-' + s for s in ['R2', 'CW_stat', 'pValue', 'Success_ratio', 'PT-stat', 'p2Value']] + \
    ['1990-' + s for s in ['R2', 'CW_stat', 'pValue', 'Success_ratio', 'PT-stat', 'p2Value']]
results_OOS_sample_forecast2.insert(0, "Forecasting models",
                                   ["HA", "SVR", "KNR", "AdaBoost", "XGBoost", "Combination"])
results_OOS_sample_forecast2.to_csv("results_OOS_sample_forecast2.csv", index=False)
results_OOS_sample_forecast2

,Forecasting models,1957-R2,1957-CW_stat,1957-pValue,1957-Success_ratio,1957-PT-stat,1957-p2Value,1990-R2,1990-CW_stat,1990-pValue,1990-Success_ratio,1990-PT-stat,1990-p2Value
0,HA,NaN,NaN,NaN,0.599739,NaN,NaN,NaN,NaN,NaN,0.641509,NaN,NaN
1,SVR,-0.619254,0.326962,0.371848,0.431551,-2.197484,0.986007,-0.854040,-0.047931,0.519115,0.431267,0.534736,0.296416
2,KNR,-0.006488,2.050013,0.020182,0.588005,2.145899,0.015941,-0.004995,1.716847,0.043004,0.606469,0.548125,0.291803
3,AdaBoost,-0.209185,0.410175,0.340839,0.474576,-1.787085,0.963038,-0.127703,0.293101,0.384723,0.409704,-1.762688,0.961023
4,XGBoost,-0.517706,1.291124,0.098330,0.518905,-0.072255,0.528801,-0.750116,0.358111,0.360130,0.533693,1.247929,0.106028
5,Combination,-0.035983,0.954144,0.170005,0.542373,1.168369,0.121329,-0.071751,0.120446,0.452065,0.574124,2.073643,0.019056
